# Brasil: SARIMA, SARIMAX, Random Forest e MLP


## 1. Dependências e configuração
.

In [ ]:
from pathlib import Path
import json, warnings
import joblib
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "datasets" / "brazil.csv").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("datasets/brazil.csv: abra o notebook dentro do projeto.")
CSV_PATH = PROJECT_ROOT / "datasets" / "brazil.csv"
OUTPUT_ROOT = PROJECT_ROOT / "outputs_brazil_comparacao"
all_predictions = {}
summary_rows = []
TARGET="goals_brazil"
TEST_SIZE=0.20
RANDOM_STATE=42

from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from itertools import product
from statsmodels.tsa.statespace.sarimax import SARIMAX
SEASONAL_PERIODS = [4, 8, 12]
MAX_CANDIDATES = 25


## 2. Limpeza e atributos compartilhados


In [ ]:
def load_and_clean(path):
    df=pd.read_csv(path).copy()
    df["date"]=pd.to_datetime(df["Date"],format="%d %b %Y",errors="coerce")
    df=df.sort_values("date",kind="stable").drop_duplicates("date",keep="first").reset_index(drop=True)
    score=df["Score"].astype("string").str.extract(r"^\s*(\d+)\s*-\s*(\d+)")
    df["home_goals"]=pd.to_numeric(score[0],errors="coerce")
    df["away_goals"]=pd.to_numeric(score[1],errors="coerce")
    df["is_home"]=df["Match"].str.startswith("Brazil v ").astype(int)
    df["opponent"]=np.where(df["is_home"].eq(1),df["Match"].str.replace(r"^Brazil v ","",regex=True),df["Match"].str.replace(r" v Brazil$","",regex=True))
    df["goals_brazil"]=np.where(df["is_home"].eq(1),df["home_goals"],df["away_goals"])
    df["goals_against"]=np.where(df["is_home"].eq(1),df["away_goals"],df["home_goals"])
    df["points"]=df["Result"].map({"W":3,"D":1,"L":0})
    df=df.dropna(subset=["date","Competition","opponent","goals_brazil","goals_against","points"]).copy()
    for c in ["goals_brazil","goals_against","points"]: df[c]=df[c].astype(float)
    return df.reset_index(drop=True)
raw=load_and_clean(CSV_PATH)
print("Linhas após limpeza:",len(raw)); display(raw.head())


In [ ]:
def create_features(df):
    data=df.copy(); data["days_since_last_match"]=data["date"].diff().dt.days.clip(lower=0)
    lags=[1,2,3,5,10]
    for lag in lags:
        data[f"goals_lag_{lag}"]=data["goals_brazil"].shift(lag)
        data[f"conceded_lag_{lag}"]=data["goals_against"].shift(lag)
        data[f"points_lag_{lag}"]=data["points"].shift(lag)
    windows=[3,5,10,20]
    g=data["goals_brazil"].shift(1); ga=data["goals_against"].shift(1); pts=data["points"].shift(1)
    for w in windows:
        data[f"goals_roll_mean_{w}"]=g.rolling(w).mean(); data[f"goals_roll_std_{w}"]=g.rolling(w).std()
        data[f"conceded_roll_mean_{w}"]=ga.rolling(w).mean(); data[f"points_roll_mean_{w}"]=pts.rolling(w).mean()
    data["month"]=data["date"].dt.month; data["month_sin"]=np.sin(2*np.pi*data["month"]/12); data["month_cos"]=np.cos(2*np.pi*data["month"]/12)
    data["day_of_year"]=data["date"].dt.dayofyear; data["doy_sin"]=np.sin(2*np.pi*data["day_of_year"]/365.25); data["doy_cos"]=np.cos(2*np.pi*data["day_of_year"]/365.25)
    data["year"]=data["date"].dt.year
    categorical_features=["opponent","Competition"]
    numeric_features=["is_home","days_since_last_match","month_sin","month_cos","doy_sin","doy_cos","year"]
    for lag in lags: numeric_features += [f"goals_lag_{lag}",f"conceded_lag_{lag}",f"points_lag_{lag}"]
    for w in windows: numeric_features += [f"goals_roll_mean_{w}",f"goals_roll_std_{w}",f"conceded_roll_mean_{w}",f"points_roll_mean_{w}"]
    feature_columns=numeric_features+categorical_features
    data=data.dropna(subset=feature_columns+[TARGET]).reset_index(drop=True)
    return data,numeric_features,categorical_features,feature_columns

data,numeric_features,categorical_features,feature_columns=create_features(raw)
split_idx=int(len(data)*(1-TEST_SIZE)); train=data.iloc[:split_idx].copy(); test=data.iloc[split_idx:].copy()
X_train=train[feature_columns]; y_train=train[TARGET]; X_test=test[feature_columns]; y_test=test[TARGET]
print("Treino:",len(train),"Teste:",len(test))


In [ ]:
def build_preprocessor():
    num=Pipeline([("imputer",SimpleImputer(strategy="median")),("scaler",StandardScaler())])
    cat=Pipeline([("imputer",SimpleImputer(strategy="most_frequent")),("onehot",OneHotEncoder(handle_unknown="ignore",sparse_output=False))])
    return ColumnTransformer([("numeric",num,numeric_features),("categorical",cat,categorical_features)])


In [ ]:
shared_data = data.copy()
shared_test_dates = test["date"].to_numpy()


## 3. Random Forest

Busca aleatória com cinco divisões temporais; seleção por MAE. O pré-processamento é ajustado dentro de cada divisão.

In [ ]:
pipeline=Pipeline([("preprocess",build_preprocessor()),("model",RandomForestRegressor(random_state=RANDOM_STATE,n_jobs=1))])
param_distributions={'model__n_estimators': [150, 300, 500, 700], 'model__max_depth': [None, 6, 10, 16], 'model__min_samples_split': [2, 5, 10], 'model__min_samples_leaf': [1, 2, 4], 'model__max_features': ['sqrt', 0.5, 1.0]}
cv=TimeSeriesSplit(n_splits=5)
search=RandomizedSearchCV(pipeline,param_distributions,n_iter=min(12,100),scoring="neg_mean_absolute_error",cv=cv,random_state=RANDOM_STATE,n_jobs=-1,refit=True,verbose=1)
search.fit(X_train,y_train)
print("Melhores hiperparâmetros:",search.best_params_)
print("MAE médio validação temporal:",round(-search.best_score_,4))


In [ ]:
def walk_forward(best_pipeline):
    preds=[]
    for i in range(len(X_test)):
        Xh=pd.concat([X_train,X_test.iloc[:i]],axis=0); yh=pd.concat([y_train,y_test.iloc[:i]],axis=0)
        m=clone(best_pipeline); m.fit(Xh,yh); preds.append(float(np.ravel(m.predict(X_test.iloc[[i]]))[0]))
    preds=np.asarray(preds)
    out=pd.DataFrame({"date":test["date"].values,"real":y_test.values,"prediction":preds,"absolute_error":np.abs(y_test.values-preds)})
    return mean_absolute_error(y_test,preds),out


In [ ]:
final_mae,predictions=walk_forward(search.best_estimator_)
print("MAE final walk-forward:",round(final_mae,4))
display(predictions.head(10))

In [ ]:
OUTPUT_DIR=OUTPUT_ROOT / "random_forest"; OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
final_model=clone(search.best_estimator_); final_model.fit(data[feature_columns],data[TARGET])
joblib.dump(final_model,OUTPUT_DIR/"modelo_final.joblib")
predictions.to_csv(OUTPUT_DIR/"walk_forward_predictions.csv",index=False)
meta={"model":"Random Forest","target":TARGET,"horizon":"1 partida","MAE_walk_forward":float(final_mae),"CV_MAE":float(-search.best_score_),"best_params":search.best_params_}
with open(OUTPUT_DIR/"config.json","w",encoding="utf-8") as f: json.dump(meta,f,ensure_ascii=False,indent=2,default=str)
print("Salvo em:",OUTPUT_DIR.resolve())

all_predictions["Random Forest"] = predictions.copy()
summary_rows.append({"modelo": "Random Forest", "MAE_teste": float(final_mae), "criterio_selecao": "MAE CV", "valor_selecao": float(-search.best_score_)})


## 4. MLP

Busca aleatória com cinco divisões temporais; seleção por MAE. O pré-processamento é ajustado dentro de cada divisão.

In [ ]:
pipeline=Pipeline([("preprocess",build_preprocessor()),("model",MLPRegressor(random_state=RANDOM_STATE,max_iter=1500,early_stopping=False))])
param_distributions={'model__hidden_layer_sizes': [(32,), (64,), (128,), (64, 32), (128, 64), (128, 64, 32)], 'model__activation': ['relu', 'tanh'], 'model__alpha': [0.0001, 0.001, 0.01, 0.1], 'model__learning_rate_init': [0.0005, 0.001, 0.005, 0.01]}
cv=TimeSeriesSplit(n_splits=5)
search=RandomizedSearchCV(pipeline,param_distributions,n_iter=min(12,100),scoring="neg_mean_absolute_error",cv=cv,random_state=RANDOM_STATE,n_jobs=-1,refit=True,verbose=1)
search.fit(X_train,y_train)
print("Melhores hiperparâmetros:",search.best_params_)
print("MAE médio validação temporal:",round(-search.best_score_,4))


In [ ]:
def walk_forward(best_pipeline):
    preds=[]
    for i in range(len(X_test)):
        Xh=pd.concat([X_train,X_test.iloc[:i]],axis=0); yh=pd.concat([y_train,y_test.iloc[:i]],axis=0)
        m=clone(best_pipeline); m.fit(Xh,yh); preds.append(float(np.ravel(m.predict(X_test.iloc[[i]]))[0]))
    preds=np.asarray(preds)
    out=pd.DataFrame({"date":test["date"].values,"real":y_test.values,"prediction":preds,"absolute_error":np.abs(y_test.values-preds)})
    return mean_absolute_error(y_test,preds),out


In [ ]:
final_mae,predictions=walk_forward(search.best_estimator_)
print("MAE final walk-forward:",round(final_mae,4))
display(predictions.head(10))

In [ ]:
OUTPUT_DIR=OUTPUT_ROOT / "mlp"; OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
final_model=clone(search.best_estimator_); final_model.fit(data[feature_columns],data[TARGET])
joblib.dump(final_model,OUTPUT_DIR/"modelo_final.joblib")
predictions.to_csv(OUTPUT_DIR/"walk_forward_predictions.csv",index=False)
meta={"model":"MLP Regressor","target":TARGET,"horizon":"1 partida","MAE_walk_forward":float(final_mae),"CV_MAE":float(-search.best_score_),"best_params":search.best_params_}
with open(OUTPUT_DIR/"config.json","w",encoding="utf-8") as f: json.dump(meta,f,ensure_ascii=False,indent=2,default=str)
print("Salvo em:",OUTPUT_DIR.resolve())

all_predictions["MLP"] = predictions.copy()
summary_rows.append({"modelo": "MLP", "MAE_teste": float(final_mae), "criterio_selecao": "MAE CV", "valor_selecao": float(-search.best_score_)})


## 5. SARIMA

Seleção das ordens pelo menor BIC no treino; avaliação de uma partida por vez no teste.

In [ ]:
data=shared_data.copy(); split_idx=int(len(data)*(1-TEST_SIZE)); train=data.iloc[:split_idx].copy(); test=data.iloc[split_idx:].copy()
y_train=train[TARGET].to_numpy(float); y_test=test[TARGET].to_numpy(float)


In [ ]:
def candidates():
    c=[]
    for p,d,q,P,D,Q,m in product([0,1,2],[0,1],[0,1,2],[0,1],[0,1],[0,1],SEASONAL_PERIODS):
        if p==d==q==P==D==Q==0: continue
        c.append(((p,d,q),(P,D,Q,m)))
    rng=np.random.default_rng(RANDOM_STATE); rng.shuffle(c); return c[:MAX_CANDIDATES]
rows=[]; best_bic=np.inf; best_order=None; best_seasonal=None
for order,seasonal in candidates():
    try:
        fit=SARIMAX(y_train,order=order,seasonal_order=seasonal,trend="c",enforce_stationarity=False,enforce_invertibility=False).fit(disp=False,maxiter=250)
        rows.append({"order":order,"seasonal_order":seasonal,"AIC":fit.aic,"BIC":fit.bic})
        if np.isfinite(fit.bic) and fit.bic<best_bic: best_bic,best_order,best_seasonal=fit.bic,order,seasonal
    except Exception: pass
if best_order is None: raise RuntimeError("Nenhum candidato produziu BIC finito; verifique os avisos de ajuste.")
search_table=pd.DataFrame(rows).sort_values("BIC").reset_index(drop=True)
print("Melhor order:",best_order,"Melhor seasonal:",best_seasonal,"BIC:",round(best_bic,2)); display(search_table.head(10))

In [ ]:
preds=[]
for i in range(len(y_test)):
    yh=np.concatenate([y_train,y_test[:i]])
    fit=SARIMAX(yh,order=best_order,seasonal_order=best_seasonal,trend="c",enforce_stationarity=False,enforce_invertibility=False).fit(disp=False,maxiter=250)
    preds.append(float(np.ravel(fit.get_forecast(steps=1).predicted_mean)[0]))
preds=np.asarray(preds); results=pd.DataFrame({"date":test["date"].values,"real":y_test,"prediction":preds,"absolute_error":np.abs(y_test-preds)}); mae=mean_absolute_error(y_test,preds)
print("MAE final walk-forward:",round(mae,4)); display(results.head(10))

In [ ]:
OUTPUT_DIR=OUTPUT_ROOT / "sarima"; OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
y_all=data[TARGET].to_numpy(float)
final=SARIMAX(y_all,order=best_order,seasonal_order=best_seasonal,trend="c",enforce_stationarity=False,enforce_invertibility=False).fit(disp=False,maxiter=250)
final.save(OUTPUT_DIR/"sarima_brazil.pkl"); results.to_csv(OUTPUT_DIR/"walk_forward_predictions.csv",index=False)
with open(OUTPUT_DIR/"config.json","w",encoding="utf-8") as f: json.dump({"order":list(best_order),"seasonal_order":list(best_seasonal),"BIC":float(best_bic),"MAE_walk_forward":float(mae)},f,ensure_ascii=False,indent=2)
print("Salvo em:",OUTPUT_DIR.resolve())
assert np.array_equal(test["date"].to_numpy(), shared_test_dates)
all_predictions["SARIMA"] = results.copy()
summary_rows.append({"modelo": "SARIMA", "MAE_teste": float(mae), "criterio_selecao": "BIC", "valor_selecao": float(best_bic)})


## 6. SARIMAX

Seleção das ordens pelo menor BIC no treino; avaliação de uma partida por vez no teste.

In [ ]:
num=["is_home","days_since_last_match","month_sin","month_cos","doy_sin","doy_cos","year"]; cat=["Competition"]; cols=num+cat
data=shared_data.dropna(subset=[TARGET]+cols).reset_index(drop=True); split_idx=int(len(data)*(1-TEST_SIZE)); train=data.iloc[:split_idx].copy(); test=data.iloc[split_idx:].copy()
pre=ColumnTransformer([("numeric",Pipeline([("imputer",SimpleImputer(strategy="median")),("scaler",StandardScaler())]),num),("categorical",Pipeline([("imputer",SimpleImputer(strategy="most_frequent")),("onehot",OneHotEncoder(handle_unknown="ignore",sparse_output=False))]),cat)])
X_train=pre.fit_transform(train[cols]); X_test=pre.transform(test[cols]); y_train=train[TARGET].to_numpy(float); y_test=test[TARGET].to_numpy(float)
print("Treino:",len(train),"Teste:",len(test))

In [ ]:
def candidates():
    c=[]
    for p,d,q,P,D,Q,m in product([0,1,2],[0,1],[0,1,2],[0,1],[0,1],[0,1],SEASONAL_PERIODS):
        if p==d==q==P==D==Q==0: continue
        c.append(((p,d,q),(P,D,Q,m)))
    rng=np.random.default_rng(RANDOM_STATE); rng.shuffle(c); return c[:MAX_CANDIDATES]
rows=[]; best_bic=np.inf; best_order=None; best_seasonal=None
for order,seasonal in candidates():
    try:
        fit=SARIMAX(y_train,exog=X_train,order=order,seasonal_order=seasonal,trend="c",enforce_stationarity=False,enforce_invertibility=False).fit(disp=False,maxiter=250)
        rows.append({"order":order,"seasonal_order":seasonal,"AIC":fit.aic,"BIC":fit.bic})
        if np.isfinite(fit.bic) and fit.bic<best_bic: best_bic,best_order,best_seasonal=fit.bic,order,seasonal
    except Exception: pass
if best_order is None: raise RuntimeError("Nenhum candidato produziu BIC finito; verifique os avisos de ajuste.")
search_table=pd.DataFrame(rows).sort_values("BIC").reset_index(drop=True)
print("Melhor order:",best_order,"Melhor seasonal:",best_seasonal,"BIC:",round(best_bic,2)); display(search_table.head(10))

In [ ]:
preds=[]
for i in range(len(y_test)):
    yh=np.concatenate([y_train,y_test[:i]]); Xh=np.vstack([X_train,X_test[:i]])
    fit=SARIMAX(yh,exog=Xh,order=best_order,seasonal_order=best_seasonal,trend="c",enforce_stationarity=False,enforce_invertibility=False).fit(disp=False,maxiter=250)
    preds.append(float(np.ravel(fit.get_forecast(steps=1,exog=X_test[i:i+1]).predicted_mean)[0]))
preds=np.asarray(preds); results=pd.DataFrame({"date":test["date"].values,"real":y_test,"prediction":preds,"absolute_error":np.abs(y_test-preds)}); mae=mean_absolute_error(y_test,preds)
print("MAE final walk-forward:",round(mae,4)); display(results.head(10))

In [ ]:
OUTPUT_DIR=OUTPUT_ROOT / "sarimax"; OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
X_all=pre.fit_transform(data[cols]); y_all=data[TARGET].to_numpy(float)
final=SARIMAX(y_all,exog=X_all,order=best_order,seasonal_order=best_seasonal,trend="c",enforce_stationarity=False,enforce_invertibility=False).fit(disp=False,maxiter=250)
final.save(OUTPUT_DIR/"sarimax_brazil.pkl"); joblib.dump(pre,OUTPUT_DIR/"exog_preprocessor.joblib"); results.to_csv(OUTPUT_DIR/"walk_forward_predictions.csv",index=False)
with open(OUTPUT_DIR/"config.json","w",encoding="utf-8") as f: json.dump({"order":list(best_order),"seasonal_order":list(best_seasonal),"BIC":float(best_bic),"MAE_walk_forward":float(mae)},f,ensure_ascii=False,indent=2)
print("Salvo em:",OUTPUT_DIR.resolve())
assert np.array_equal(test["date"].to_numpy(), shared_test_dates)
all_predictions["SARIMAX"] = results.copy()
summary_rows.append({"modelo": "SARIMAX", "MAE_teste": float(mae), "criterio_selecao": "BIC", "valor_selecao": float(best_bic)})


## 7. Comparação final

In [ ]:
comparison = pd.DataFrame(summary_rows).sort_values("MAE_teste").reset_index(drop=True)
display(comparison)
combined = pd.DataFrame({"date": shared_test_dates, "real": all_predictions["Random Forest"]["real"].to_numpy()})
for name, predictions_model in all_predictions.items():
    assert np.array_equal(predictions_model["date"].to_numpy(), shared_test_dates)
    combined[name] = predictions_model["prediction"].to_numpy()
display(combined.head(10))
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
comparison.to_csv(OUTPUT_ROOT / "comparacao_modelos.csv", index=False)
combined.to_csv(OUTPUT_ROOT / "previsoes_comparadas.csv", index=False)
print("Resultados salvos em:", OUTPUT_ROOT)
